# 00 – Get Data from the data.gov.sg API
**Purpose:** Retrieve HDB resale transactions and the HDB Resale Price Index through an authenticated REST API, save the raw responses, and document the response schema (RR-P1 Task 1).
**Input:** `DATA_GOV_SG_API_KEY` in `.env` · reusable client `src/api_client.py`
**Output:**
- `data/raw/resale_pages/*.json` – every raw API page (reproducible, used to resume)
- `data/raw/hdb_resale_raw.csv` – all resale transactions, unchanged
- `data/raw/rpi_pages/*.json`, `data/raw/hdb_rpi_raw.csv` – Resale Price Index, unchanged
- `data/raw/retrieval_log.json` – what was retrieved, when, how many calls

**Issues:** #4 Retrieve resale transactions · #5 Retrieve Resale Price Index · #6 Document endpoints and schema (copy the printed schema into `docs/api_endpoints.md`)

> Raw files are never edited. Cleaning happens in `02_clean.ipynb`.

In [ ]:
# Imports and paths
import sys, json
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()                       # repo root (this notebook lives in notebooks/)
sys.path.insert(0, str(ROOT / 'src'))             # so we can import src/api_client.py
from api_client import DataGovClient, BASE_URL

RAW_DIR = ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

RESALE_ID = 'd_8b84c4ee58e3cfc0ece0d773c8ca6abc'  # Resale flat prices, Jan 2017 onwards
RPI_ID    = 'd_14f63e595975691e7c24a27ae4c07c79'  # HDB Resale Price Index (1Q2009 = 100), quarterly

## 1. Authenticate
The client reads the key from `.env` and sends it in the **`x-api-key`** request header. Without a key the API still answers, but with a much lower rate limit (4 instead of 8 calls per 10 seconds for a dev key).

In [ ]:
# Create the client (key comes from .env - it is never printed)
client = DataGovClient(min_interval=1.3)          # 1.3 s between calls -> under 8 calls / 10 s
print('API key loaded:', client.authenticated)
assert client.authenticated, "No key found - copy .env.example to .env and paste your data.gov.sg API key"

## 2. Document the endpoint (issue #6)

In [ ]:
# Base URL, headers (key masked) and query parameters used
print('Base URL :', BASE_URL)
print('Method   : GET')
print('Headers  :', client.headers_for_docs())
print('Params   : resource_id, limit (rows per page), offset (rows to skip), sort (_id asc for stable paging)')

In [ ]:
# One small call to see the full response structure
sample = client.get_page(RESALE_ID, limit=3)
print('Top-level keys :', list(sample.keys()))
print('result keys    :', list(sample['result'].keys()))
print('Total rows     :', f"{sample['result'].get('total'):,}")
(RAW_DIR / 'resale_sample_response.json').write_text(json.dumps(sample, indent=2), encoding='utf-8')  # keep an example response
pd.DataFrame(sample['result']['records'])

In [ ]:
# Response schema: field names and API data types -> copy into docs/api_endpoints.md
fields, total = client.schema(RESALE_ID)
print(f'{len(fields)} fields, {total:,} rows')
fields

## 3. Retrieve all resale transactions (issue #4)
Pages of up to 10,000 rows are requested; if the API returns fewer, the client simply moves on by what it received.
Every page is saved first, so **if the run stops (rate limit, Wi-Fi), just run this cell again** – saved pages are read from disk and only the missing ones are called.

In [ ]:
# Fetch every page (takes a few minutes; progress is printed)
resale_records = client.fetch_all(RESALE_ID, page_size=10000, raw_dir=RAW_DIR / 'resale_pages')
resale = pd.DataFrame(resale_records)
print(resale.shape)
resale.head()

In [ ]:
# Completeness checks: every row once, count matches the API total
api_total = client.schema(RESALE_ID)[1]
print('Rows retrieved :', f'{len(resale):,}')
print('API total      :', f'{api_total:,}')
print('Unique _id     :', f"{resale['_id'].nunique():,}")
print('Month range    :', resale['month'].min(), 'to', resale['month'].max())
assert len(resale) == resale['_id'].nunique(), 'Duplicate _id - paging overlapped'
if len(resale) != api_total:
    print('NOTE: row count differs from API total - new rows may have been added during the run; re-run to refresh.')

In [ ]:
# Save the combined raw table (unchanged values, all text as delivered by the API)
resale.to_csv(RAW_DIR / 'hdb_resale_raw.csv', index=False)
print('Saved', RAW_DIR / 'hdb_resale_raw.csv')

## 4. Retrieve the HDB Resale Price Index (issue #5)

In [ ]:
# Small dataset - same client, same paging
rpi_fields, rpi_total = client.schema(RPI_ID)
print(f'{len(rpi_fields)} fields, {rpi_total} rows')
display(rpi_fields)

rpi = client.fetch_all_df(RPI_ID, page_size=1000, raw_dir=RAW_DIR / 'rpi_pages')
rpi.to_csv(RAW_DIR / 'hdb_rpi_raw.csv', index=False)
print(rpi.shape)
rpi.tail()

## 5. Retrieval log (evidence for the Deliverables Checklist)

In [ ]:
# Record what was retrieved and when
log = {
    'retrieved_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'base_url': BASE_URL,
    'auth': 'x-api-key header (key in .env, not committed)',
    'api_calls_this_run': client.calls,
    'datasets': {
        'resale': {'resource_id': RESALE_ID, 'rows': len(resale), 'api_total': api_total,
                   'month_min': resale['month'].min(), 'month_max': resale['month'].max(),
                   'raw_csv': 'data/raw/hdb_resale_raw.csv', 'raw_pages': 'data/raw/resale_pages/'},
        'rpi': {'resource_id': RPI_ID, 'rows': len(rpi),
                'raw_csv': 'data/raw/hdb_rpi_raw.csv', 'raw_pages': 'data/raw/rpi_pages/'},
    },
}
(RAW_DIR / 'retrieval_log.json').write_text(json.dumps(log, indent=2), encoding='utf-8')
print(json.dumps(log, indent=2))

## Next
- Copy the schema tables above into `docs/api_endpoints.md` (issue #6).
- Commit the notebook and `src/api_client.py` – **not** the data (`data/raw` is in `.gitignore`; teammates re-run this notebook).
- Continue with `01_inspect.ipynb`.